## Program 1E

Reimplementation of program 1E, but with spatial hashing check peformed with a direct AABB against C2, instead of S1. Cell size remains dictated by the radius of S1.

In [1]:
n_rows = 1

In [2]:
import pandas as pd
import numpy as np
import time
from time import perf_counter_ns
from collections import defaultdict

h_1 = 20
r_1 = 20
h_2 = 10
r_2 = 20

file_path = r"C:\Users\Smith\OneDrive\MSc Project\05 Final Code\Bunny Head Raw Lines for Computation Testing.xlsx"

df = pd.read_excel(
    file_path,
    sheet_name=0,
    usecols="A:F",
    nrows=n_rows,
    header=None,
    engine="openpyxl"
)

df.columns = ["x", "y", "z", "i", "j", "k"]
A = df.to_numpy(dtype=float)

In [3]:
def point_in_CTC_AABB_spatial_hash_chunked(A, h_1, r_1, h_2, r_2, eps=1e-12, row_block=256, pair_block=250_000):

    t_start = perf_counter_ns()
    t_last = t_start

    def lap(name):
        nonlocal t_last
        now = perf_counter_ns()
        dt_ms = (now - t_last) / 1_000_000
        total_ms = (now - t_start) / 1_000_000
        print(f"{name:<45} /{dt_ms:10.3f}/ ms   total: {total_ms:10.3f} ms")
        t_last = now

    # Positive integer check for S1 theory
    if h_1 <= 0 or h_2 <= 0 or r_1 <= 0 or r_2 <= 0:
        raise ValueError("h1, h2, r1 and r2 must be positive values.")

    # Radius comparison check
    if r_2 < r_1:
        raise ValueError("r2 < r1.")

    A = np.asarray(A, dtype=np.float64)
    lap("Input to numpy array")

    P = np.ascontiguousarray(A[:, 0:3])   # xyz points
    U = np.ascontiguousarray(A[:, 3:6])   # orientation vectors
    N = A.shape[0]
    lap("Separate Points and Vectors")

    H = h_1 + h_2

    k_sphere = max(
        (r_2*r_2 + h_1*h_1) / (2.0*h_1),
        (r_2*r_2 + H*H) / (2.0*H)
    )

    R = k_sphere
    R_sq = (R + eps) * (R + eps)

    R_query = R + eps

    cell_size = R
    inv_cell_size = 1.0 / cell_size

    lap("Create S1")

    grid = defaultdict(list)

    point_cell_keys = np.floor(P * inv_cell_size).astype(np.int64)

    for idx in range(N):
        grid[tuple(point_cell_keys[idx])].append(idx)

    for key in list(grid.keys()):
        grid[key] = np.asarray(grid[key], dtype=np.int64)

    hit_chunks = []

    broad_cell_candidate_count = 0
    envelope_aabb_candidate_count = 0
    sphere_candidate_count = 0
    exact_test_count = 0

    def process_pair_arrays(pair_i, pair_j):

        nonlocal exact_test_count

        M = pair_i.size

        for p0 in range(0, M, pair_block):
            p1 = min(p0 + pair_block, M)

            I = pair_i[p0:p1]
            J = pair_j[p0:p1]

            Pi = P[I]
            Ui = U[I]
            Pj = P[J]

            V = Pj - Pi

            # t = distance along the current CTC centreline
            t = np.einsum("ij,ij->i", V, Ui)

            # squared distance from tip to point
            v2 = np.einsum("ij,ij->i", V, V)

            # squared perpendicular distance to centreline
            d_perp_sq = v2 - t*t
            d_perp_sq = np.maximum(d_perp_sq, 0.0)

            exact_test_count += I.size

            # slab test
            axial_ok = (t >= -eps) & (t <= H + eps)

            # Cone/cylinder radial test
            cone_region = t <= h_1 + eps
            cylinder_region = t > h_1 + eps

            cone_ok = (h_1*h_1*d_perp_sq) <= (r_1*r_1*t*t + eps)
            cylinder_ok = d_perp_sq <= (r_2*r_2 + eps)

            radial_ok = (cone_region & cone_ok) | (cylinder_region & cylinder_ok)

            ok = axial_ok & radial_ok

            if np.any(ok):
                hits = np.column_stack((I[ok], J[ok]))
                hit_chunks.append(hits)

    # The New Big Loop!
    loop_start = perf_counter_ns()

    for i0 in range(0, N, row_block):
        i1 = min(i0 + row_block, N)

        block_pair_i_chunks = []
        block_pair_j_chunks = []
        block_pair_count = 0

        for i in range(i0, i1):

            tip = P[i]
            axis = U[i]
            end = tip + H * axis

            envelope_expand = r_2 * np.sqrt(np.maximum(1.0 - axis*axis, 0.0)) + eps

            envelope_min = np.minimum(tip, end) - envelope_expand
            envelope_max = np.maximum(tip, end) + envelope_expand

            # Cells overlapping C2 envelope AABB
            cmin = np.floor(envelope_min * inv_cell_size).astype(np.int64)
            cmax = np.floor(envelope_max * inv_cell_size).astype(np.int64)

            candidate_chunks = []

            for cx in range(cmin[0], cmax[0] + 1):
                for cy in range(cmin[1], cmax[1] + 1):
                    for cz in range(cmin[2], cmax[2] + 1):

                        ids = grid.get((cx, cy, cz))

                        if ids is not None:
                            candidate_chunks.append(ids)

            if not candidate_chunks:
                continue

            J = np.concatenate(candidate_chunks)

            broad_cell_candidate_count += J.size

            # Apply new triangular mask j < i (for whole consolidated pair list, not the NxN)
            J = J[J < i]

            if J.size == 0:
                continue

            Pj = P[J]

            #AABB check alternative
            envelope_aabb_ok = (
                (Pj[:, 0] >= envelope_min[0]) & (Pj[:, 0] <= envelope_max[0]) &
                (Pj[:, 1] >= envelope_min[1]) & (Pj[:, 1] <= envelope_max[1]) &
                (Pj[:, 2] >= envelope_min[2]) & (Pj[:, 2] <= envelope_max[2])
            )

            J = J[envelope_aabb_ok]

            envelope_aabb_candidate_count += J.size

            if J.size == 0:
                continue

            I = np.full(J.size, i, dtype=np.int64)

            block_pair_i_chunks.append(I)
            block_pair_j_chunks.append(J)
            block_pair_count += J.size

            # Periodic clearance to prevent very large candidate buffers
            if block_pair_count >= pair_block:
                pair_i = np.concatenate(block_pair_i_chunks)
                pair_j = np.concatenate(block_pair_j_chunks)

                process_pair_arrays(pair_i, pair_j)

                block_pair_i_chunks.clear()
                block_pair_j_chunks.clear()
                block_pair_count = 0

        # Clearance of remaining pairs from row block
        if block_pair_count:
            pair_i = np.concatenate(block_pair_i_chunks)
            pair_j = np.concatenate(block_pair_j_chunks)

            process_pair_arrays(pair_i, pair_j)

    loop_ms = (perf_counter_ns() - loop_start) / 1_000_000
    lap("Spatial hash broad phase and exact tests")

    if hit_chunks:
        hit_pairs = np.vstack(hit_chunks)
    else:
        hit_pairs = np.empty((0, 2), dtype=np.int64)

    lap("Hit list")

    print("-" * 75)
    print(f"Sphere centre offset k / radius R: {k_sphere:.6g}")
    print(f"Cell size:                       {cell_size:.6g}")
    print(f"Broad cell candidates:           {broad_cell_candidate_count:,}")
    print(f"Envelope AABB candidates:        {envelope_aabb_candidate_count:,}")
    print(f"Intersection tests:              {exact_test_count:,}")
    print(f"Hit pairs:                       {hit_pairs.shape[0]:,}")
    print(f"{'TOTAL':<45} {(perf_counter_ns() - t_start) / 1_000_000:10.3f} ms")
    print(f"Row block:                       {row_block:,}")
    print(f"Pair block:                      {pair_block:,}")

    return hit_pairs

In [4]:
start = time.perf_counter()
hit_pairs = point_in_CTC_AABB_spatial_hash_chunked(A, h_1, r_1, h_2, r_2, row_block=256, pair_block=250_000)
elapsed_ms = (time.perf_counter() - start) * 1000
print(f"Function time: {elapsed_ms:.3f} ms")
print(hit_pairs.shape)

Input to numpy array                          /     0.020/ ms   total:      0.020 ms
Separate Points and Vectors                   /     0.315/ ms   total:      0.335 ms
Create S1                                     /     0.059/ ms   total:      0.395 ms
Spatial hash broad phase and exact tests      /     2.815/ ms   total:      3.209 ms
Hit list                                      /     0.144/ ms   total:      3.354 ms
---------------------------------------------------------------------------
Sphere centre offset k / radius R: 21.6667
Cell size:                       21.6667
Broad cell candidates:           1
Envelope AABB candidates:        0
Intersection tests:              0
Hit pairs:                       0
TOTAL                                              3.639 ms
Row block:                       256
Pair block:                      250,000
Function time: 4.077 ms
(0, 2)
